## Installing Dependencies day 02

RAG:document loading
embedding models
vector databases
llm model

In [1]:
print("Installing Dependencies")

!pip install -q \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-groq \
    langchain-google-genai \
    langchain-openai \
    langchain-core \
    faiss-cpu \
    pypdf \
    sentence-transformers \
    transformers \
    torch \
    huggingface_hub \
    groq \
    langsmith \
    python-dotenv \
    tiktoken
print('\Installation completed')


Installing Dependencies
\Installation completed


In [2]:
LLM_Provider = 'groq'
LLM_Model = 'openai/gpt-oss-120b'

In [3]:
import os 
from dotenv import load_dotenv
load_dotenv()
if os.getenv('Groq_API_KEY') :
    print("Groq API KEY is imported!")
else:
    print("Groq API KEY is not imported!")
from langchain_community.document_loaders import PyPDFDirectoryLoader   #<--loading data
from langchain_text_splitters import RecursiveCharacterTextSplitter        #<--splitting data into chunks
             #<-- prompt writtens as per users query

Groq API KEY is imported!


C:\Users\varshitha\AppData\Local\Temp\ipykernel_7644\101855686.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader   #<--loading data


In [4]:
CORPUS_PATH = "./zyro-dynamics-hr-corpus" #<--path for kb
loader= PyPDFDirectoryLoader(CORPUS_PATH)      #<--loading the documents from the path
documents= loader.load()                      
print(f"Loaded {len(documents)} documents")   #<--printing the number of documents loaded..........also len gives us no of pages

Loaded 39 documents


In [5]:
splitter= RecursiveCharacterTextSplitter(
    chunk_size=800,    #<--RecursiveCharacterTextSplitter
    chunk_overlap=100  #<--overlap between chunks to maintain context
)
chunks= splitter.split_documents(documents)  #<--splitting the documents into chunks
print(f"Created {len(chunks)} chunks")  #<--printing the number of chunks created

Created 107 chunks


## Embeddings Initialisation day 03
HuggingFaceEmbeddings

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings  #<--"I want to use the Hugging Face embedding component provided through the LangChain ecosystem."
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("Embeddings Done")
#<--only the initialising for embeddings are done not the actual embedding of the chunks is done yet because we have not created the vector store yet.............after storing data we can apply embeddings on it
#<--creating the vector store using FAISS(some of the, are chromadb quant ets) 

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings Done


## VectorDB + Retrival
FAISS Vector DB

In [7]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(chunks, embedding_model)  #<--creating the vector store using FAISS .......also (chunk -- chunk of data.....model-- embedding model)
#Retrival
retriever = vectorstore.as_retriever(search_kwargs={ "k": 3})  #<--retrieving the data from the vector store using retriever
print("Done!")

Done!


## LLM initialisation 
LLM model = groq


In [ ]:

from langchain_groq import ChatGroq
llm_model = ChatGroq(
model='openai/gpt-oss-120b',
temperature = 0.1 ,  #<--LEVEL OF RANDOMNESS IN OUTPUT PROMPT GENERATED(sentence getting formed is controlled by temperature)
max_tokens = 500
)
print(f"LLM_Model groq initialised successfully")

LLM_Model groq initialised successfully


## RAG CHAIN 
Initialization


In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langsmith import traceable
RAG_PROMPT= ChatPromptTemplate.from_template(
    ## define rag prompt here
 """
  You are an HR Assistant at zyro dynamics.
Answer the questions asked by employees by only using the knowledge base which are given . 
If the suitable answer isn't in the knowledge base , say you dont have that information.

context: {context},   #<--all the info required for LLM Model to get the output
Question: {question}
"""
)


def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])  #<--formatting the retrieved documents into a string format to be used in the prompt

@traceable(name="rag_chain")
def rag_chain(question: str):     #<--rag_chain is a function which takes question as input and returns the answer
     docs = retriever.invoke(question)  #<--retrieving the relevant documents from the vector store using retriever(checks specific question in the vector store and returns the relevant documents)-->invoke is to seacrhing the question in vectordb
     context = format_docs(docs)    #<--formatting the retrieved documents into a string format to be used in the prompt
     chain= RAG_PROMPT | llm_model | StrOutputParser() #<--creating a chain of prompt and llm model to generate the answer---> | means chain of steps
     answer = chain.invoke({"context": context, "question": question})  #<--invoking the chain with the context and question to get the answer
     return {"answer": answer, "source": docs}  #<--returning the answer and context as a dictionary

## Guardials
To ensure no context is used beyound corpus

In [10]:
GUARDIAL_PROMPT = ChatPromptTemplate.from_template(
    ## define guardial prompt here
 """
  You are a scope classifier for HR assisstant. decide whether the question below is somethong an HR assisstant should answer (company leave policy, reimbursment, code of conduct, or other internal HR topics) or something out of scope (general knowledge , coding help, unrelated small talk, etc). Question: {question}""")
REFUSAL_MESSAGE=(
    "I'm an HR assisstant and can only help with questions about company HR policies" 
    "policies(leave, reimbursement, code of conduct etc.) , I don't have " 
    "information to answer that question."
)

def ask_bot(question: str):
    guardial_chain = GUARDIAL_PROMPT | llm_model | StrOutputParser()
    verdict = guardial_chain.invoke({"question": question}).strip().upper()
    if 'OUT_OF_SCOPE' in verdict:
        return {"answer": REFUSAL_MESSAGE, "source": []}
    return rag_chain(question)
print("Guardials Initialized!")

Guardials Initialized!


In [11]:
test_question =[
    "How many casual leaves do I get in a year?",  #<--related to corpus
    "What is the internet reimbursement limit?", #<--related to corpus
    "what's the capital of France? ",   #<--not related to corpus
    ]
for i,q in enumerate(test_question, 1):
    result = ask_bot(q)
    print(f"Q{i}: {q}")
    print(f"A{i}: {result['answer']}")
    if result['source']:
        print(
              f"Source:"
              f"{[d.metadata.get('source') for d in result['source']]}"
        )
    print("-"*60)
        
    

Q1: How many casual leaves do I get in a year?
A1: I’m sorry, but I don’t have that information.
Source:['zyro-dynamics-hr-corpus\\02_Leave_Policy.pdf', 'zyro-dynamics-hr-corpus\\02_Leave_Policy.pdf', 'zyro-dynamics-hr-corpus\\06_Compensation_and_Benefits_Policy.pdf']
------------------------------------------------------------
Q2: What is the internet reimbursement limit?
A2: The internet reimbursement is **Rs. 1,000 per month**, and it is available only to Full‑Remote employees at level 5 or above.
Source:['zyro-dynamics-hr-corpus\\03_Work_From_Home_Policy.pdf', 'zyro-dynamics-hr-corpus\\07_IT_and_Data_Security_Policy.pdf', 'zyro-dynamics-hr-corpus\\10_Travel_and_Expense_Policy.pdf']
------------------------------------------------------------
Q3: what's the capital of France? 
A3: I’m sorry, but I don’t have that information.
Source:['zyro-dynamics-hr-corpus\\06_Compensation_and_Benefits_Policy.pdf', 'zyro-dynamics-hr-corpus\\00_Company_Profile.pdf', 'zyro-dynamics-hr-corpus\\05_Per

In [ ]:
import pandas as pd
test_df = pd.read_csv("test.csv")
test_df
answers = []
for i,row in test_df.iterrows():
    q = row["question"]
    result = ask_bot(q)
    answers.append(result["answer"])
    print(f"[{i+1}/len{len(test_df)}] {q}\n-> {result['answer']}\n{'-'*60}")

test_df["answer"] = answers
submission = test_df[["question_id", "answer"]]
submission.to_csv("submission.csv", index=False)
submission.head()


[1/len20] How does my Earned Leave accrue every month?
-> Earned Leave (EL) builds up each month as follows:

- **After completing your first year of continuous service (with at least 240 days worked)** you become eligible for a block of 15 days of EL.  
- **From that point onward, EL accrues at **`1.25 days per month`.  

- **If you are still in your probation period**, EL accrues at a reduced rate of **0.5 days per month**; these days become usable only after your probation is confirmed.  

So, once you’re a regular employee (post‑probation and past the first‑year milestone), you add **1.25 EL days each month** to your balance.
------------------------------------------------------------
[2/len20] How much Earned Leave can I carry forward to next year?
-> I’m sorry, but I don’t have that information.
------------------------------------------------------------
[3/len20] How many weeks of Maternity Leave am I entitled to?
-> You are entitled to **26 weeks of paid Maternity Leave** for

,question_id,answer
0,Q01,Earned Leave (EL) builds up each month as foll...
1,Q02,"I’m sorry, but I don’t have that information."
2,Q03,You are entitled to **26 weeks of paid Materni...
3,Q04,"Yes. According to Zyro Dynamics’ leave policy,..."
4,Q05,Your salary (and any professional fees) is pro...
